In [11]:
import ast
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [15]:
RANDOM_STATE = 42
MIN_FILMS = 50
MIN_NONZERO = 20

TOP_KW_INTERACTIONS = 15

ALPHA_VALUES = [
    500,
    1000,
    2000,
    3000,
    5000,
    8000,
    10000,
]

AUTO_SELECT_BEST = True

FORCED_MODEL_NAME = "D_text_full_interactions"
FORCED_ALPHA = 2000.0


def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        marker_file = candidate / "analysis" / "data" / "Lumiere_Country_Full_Backup.csv"

        if marker_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Make sure the notebook is inside the Analytics2Action project folder."
    )


REPO_ROOT = find_repo_root(Path.cwd())

DATA_DIR = REPO_ROOT / "analysis" / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"

ADMISSIONS_PATH = DATA_DIR / "Lumiere_Country_Full_Backup.csv"
FINANCIALS_PATH = DATA_DIR / "IMDB_financials.xlsx"
CLUSTERS_PATH = DATA_DIR / "clustered_films_k6.csv"

TEXT_SVD_PATH = OUTPUT_DIR / "text_numeric_features" / "tfidf_svd_features_50.csv"

COMPARISON_OUTPUT_PATH = OUTPUT_DIR / "model_comparison_with_text_svd.csv"
EXPORT_OUTPUT_PATH = OUTPUT_DIR / "model_export_with_text_svd.json"
PREDICTIONS_OUTPUT_PATH = OUTPUT_DIR / "model_predictions_with_text_svd.csv"

print("Current working directory:", Path.cwd())
print("Detected repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

print("\nChecking files:")
print("Admissions:", ADMISSIONS_PATH.exists(), ADMISSIONS_PATH.relative_to(REPO_ROOT))
print("Financials:", FINANCIALS_PATH.exists(), FINANCIALS_PATH.relative_to(REPO_ROOT))
print("Clusters:", CLUSTERS_PATH.exists(), CLUSTERS_PATH.relative_to(REPO_ROOT))
print("Text SVD:", TEXT_SVD_PATH.exists(), TEXT_SVD_PATH.relative_to(REPO_ROOT))

missing_files = []

if not ADMISSIONS_PATH.exists():
    missing_files.append(ADMISSIONS_PATH.relative_to(REPO_ROOT))

if not FINANCIALS_PATH.exists():
    missing_files.append(FINANCIALS_PATH.relative_to(REPO_ROOT))

if not CLUSTERS_PATH.exists():
    missing_files.append(CLUSTERS_PATH.relative_to(REPO_ROOT))

if not TEXT_SVD_PATH.exists():
    missing_files.append(TEXT_SVD_PATH.relative_to(REPO_ROOT))

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(str(path) for path in missing_files)
    )

Current working directory: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\notebooks\text-numeric-features
Detected repo root: C:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action
Data dir: C:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\analysis\data
Output dir: C:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs

Checking files:
Admissions: True analysis\data\Lumiere_Country_Full_Backup.csv
Financials: True analysis\data\IMDB_financials.xlsx
Clusters: True analysis\data\clustered_films_k6.csv
Text SVD: True outputs\text_numeric_features\tfidf_svd_features_50.csv


In [16]:
df_admissions = pd.read_csv(ADMISSIONS_PATH)
df_clusters = pd.read_csv(CLUSTERS_PATH)
df_text_svd = pd.read_csv(TEXT_SVD_PATH)

budget_both = pd.read_excel(FINANCIALS_PATH, sheet_name="Both")
budget_only = pd.read_excel(FINANCIALS_PATH, sheet_name="Budget only")

df_budget = pd.concat([budget_both, budget_only], ignore_index=True)
df_budget = df_budget[["titleId", "budget_eur"]].copy()
df_budget["budget_eur"] = pd.to_numeric(df_budget["budget_eur"], errors="coerce")
df_budget = df_budget.dropna(subset=["budget_eur"])
df_budget = df_budget.drop_duplicates(subset=["titleId"])

if "titleId" in df_clusters.columns:
    df_clusters_small = df_clusters[["titleId", "cluster"]].drop_duplicates(subset=["titleId"])
    df = df_admissions.merge(df_clusters_small, on="titleId", how="inner")
else:
    if len(df_admissions) != len(df_clusters):
        raise ValueError(
            "Clusters file has no titleId and row count does not match admissions file."
        )

    df = df_admissions.copy()
    df["cluster"] = df_clusters["cluster"].values

df = df.merge(df_budget, on="titleId", how="inner")
df = df.merge(df_text_svd, on="titleId", how="left")

text_svd_cols = [
    col for col in df.columns
    if col.startswith("text_svd_")
]

df[text_svd_cols] = df[text_svd_cols].fillna(0)

print("Admissions films:", len(df_admissions))
print("Cluster rows:", len(df_clusters))
print("Films with budget:", len(df_budget))
print("Text SVD rows:", len(df_text_svd))
print("Merged films:", len(df))
print("Text SVD features:", len(text_svd_cols))

Admissions films: 2000
Cluster rows: 2000
Films with budget: 1107
Text SVD rows: 2000
Merged films: 1107
Text SVD features: 50


In [17]:
def parse_country_admissions(value):
    if pd.isna(value) or value == "Not Found":
        return {}

    if isinstance(value, dict):
        return value

    value = str(value)

    try:
        parsed = json.loads(value.replace('""', '"'))
        if isinstance(parsed, dict):
            return parsed
    except (json.JSONDecodeError, TypeError):
        pass

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, dict):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return {}


def parse_admission_count(value):
    if pd.isna(value):
        return np.nan

    value = str(value).replace(",", "").strip()
    return pd.to_numeric(value, errors="coerce")


def parse_keywords(value):
    if pd.isna(value) or value == "":
        return []
    return [k.strip().lower() for k in str(value).split(",") if k.strip()]


def parse_genres(value):
    if pd.isna(value) or value == "":
        return []
    return [g.strip().lower() for g in str(value).split(",") if g.strip()]


def is_domestic_market(origin_country, market_country):
    origin = str(origin_country).strip().upper()
    market = str(market_country).strip().upper()

    if market == origin:
        return True

    if market in ["GBIE", "GB_IE"] and origin in ["GB", "IE"]:
        return True

    return False


def drop_sparse_cols(dataframe, min_nonzero=MIN_NONZERO):
    keep = dataframe.columns[(dataframe != 0).sum(axis=0) >= min_nonzero]
    return dataframe[keep]

In [18]:
records = []

for _, row in df.iterrows():
    country_admissions = parse_country_admissions(row["country_admissions"])

    for country, count in country_admissions.items():
        if str(country).startswith("Total"):
            continue

        record = {
            "titleId": row["titleId"],
            "originalTitle": row["originalTitle"],
            "releaseYear": row["releaseYear"],
            "mainCountry": row["mainCountry"],
            "firstLanguage": row["firstLanguage"],
            "genres": row["genres"],
            "keywords": row["keywords"],
            "cluster": row["cluster"],
            "budget_eur": row["budget_eur"],
            "country": country,
            "admissions": parse_admission_count(count),
        }

        for col in text_svd_cols:
            record[col] = row[col]

        records.append(record)

df_long = pd.DataFrame(records)

df_long = df_long.dropna(subset=["admissions"])
df_long = df_long[df_long["admissions"] >= 0].copy()

print("Long rows before filtering:", len(df_long))
print("Unique films before filtering:", df_long["titleId"].nunique())
print("Unique countries before filtering:", df_long["country"].nunique())

Long rows before filtering: 16350
Unique films before filtering: 1064
Unique countries before filtering: 40


In [19]:
country_film_counts = df_long.groupby("country")["titleId"].nunique()
small_countries = country_film_counts[country_film_counts < MIN_FILMS].index.tolist()

df_long = df_long[~df_long["country"].isin(small_countries)].copy()

films_with_gbie = set(
    df_long.loc[
        df_long["country"].isin(["GBIE", "GB_IE"]),
        "titleId"
    ]
)

mask_redundant = (
    df_long["country"].isin(["GB", "IE"]) &
    df_long["titleId"].isin(films_with_gbie)
)

df_long = df_long[~mask_redundant].copy()

df_long["is_domestic"] = df_long.apply(
    lambda row: is_domestic_market(row["mainCountry"], row["country"]),
    axis=1
)

df_long["log_admissions"] = np.log1p(df_long["admissions"])

country_mean = df_long.groupby("country")["log_admissions"].transform("mean")
country_std = df_long.groupby("country")["log_admissions"].transform("std")

df_long["admissions_zscore"] = (
    df_long["log_admissions"] - country_mean
) / country_std

df_long = df_long.replace([np.inf, -np.inf], np.nan)
df_long = df_long.dropna(subset=["admissions_zscore"]).copy()

df_long["budget_bracket"] = pd.cut(
    df_long["budget_eur"],
    bins=[0, 3_000_000, 8_000_000, 25_000_000, float("inf")],
    labels=[
        "micro (<3M)",
        "european (3-8M)",
        "mid (8-25M)",
        "big (>25M)"
    ]
)

print("Long rows after filtering:", len(df_long))
print("Unique films after filtering:", df_long["titleId"].nunique())
print("Unique countries after filtering:", df_long["country"].nunique())

print("\nBudget bracket distribution:")
print(df_long.groupby("budget_bracket")["titleId"].nunique())

Long rows after filtering: 15870
Unique films after filtering: 1064
Unique countries after filtering: 35

Budget bracket distribution:
budget_bracket
micro (<3M)        279
european (3-8M)    197
mid (8-25M)        322
big (>25M)         265
Name: titleId, dtype: int64


C:\Users\Georg\AppData\Local\Temp\ipykernel_23368\2811013116.py:53: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_long.groupby("budget_bracket")["titleId"].nunique())


In [20]:
df_long["keywords_list"] = df_long["keywords"].apply(parse_keywords)
df_long["genres_list"] = df_long["genres"].apply(parse_genres)

keyword_film_counts = (
    df_long.drop_duplicates("titleId")
    .explode("keywords_list")["keywords_list"]
    .value_counts()
)

valid_keywords_50 = set(
    keyword_film_counts[keyword_film_counts >= 50].index
)

df_kw = df_long.explode("keywords_list")
df_kw = df_kw[df_kw["keywords_list"].isin(valid_keywords_50)].copy()

kw_country = (
    df_kw.groupby(["keywords_list", "country"])["admissions_zscore"]
    .agg(["mean", "count"])
    .reset_index()
)

kw_country = kw_country[kw_country["count"] >= 10].copy()

kw_variance = (
    kw_country.groupby("keywords_list")["mean"]
    .agg(["std", "count"])
    .rename(columns={"count": "n_countries"})
)

kw_variance = (
    kw_variance[kw_variance["n_countries"] >= 15]
    .sort_values("std", ascending=False)
)

valid_keywords_base = valid_keywords_50
top_divisive_kw = kw_variance.head(TOP_KW_INTERACTIONS).index.tolist()

print("Base keywords with 50+ films:", len(valid_keywords_base))
print("Top divisive interaction keywords:")
print(top_divisive_kw)

Base keywords with 50+ films: 470
Top divisive interaction keywords:
['blockbuster', 'villain', 'hero', 'british', 'evil', 'evil man', 'cult film', 'underwear', 'action hero', 'writer', 'cruelty', 'laughter', 'femme fatale', 'transformation', 'good versus evil']


In [21]:
y = df_long["admissions_zscore"].values

market_dummies = pd.get_dummies(df_long["country"], prefix="market")
origin_dummies = pd.get_dummies(df_long["mainCountry"], prefix="origin")
lang_dummies = pd.get_dummies(df_long["firstLanguage"], prefix="lang")
cluster_dummies = pd.get_dummies(df_long["cluster"], prefix="cluster")
domestic_flag = df_long["is_domestic"].astype(int).to_frame("is_domestic")
bracket_dummies = pd.get_dummies(df_long["budget_bracket"], prefix="budget")

all_genres = set()

for g_list in df_long["genres_list"]:
    all_genres.update(g_list)

genre_dummies = pd.DataFrame(
    0,
    index=df_long.index,
    columns=[f"genre_{g}" for g in sorted(all_genres)],
    dtype=np.int8
)

for row_index, g_list in zip(df_long.index, df_long["genres_list"]):
    for g in g_list:
        col = f"genre_{g}"
        if col in genre_dummies.columns:
            genre_dummies.loc[row_index, col] = 1

kw_cols = sorted([f"kw_{k}" for k in valid_keywords_base])

keyword_features = pd.DataFrame(
    0,
    index=df_long.index,
    columns=kw_cols,
    dtype=np.int8
)

for row_index, kw_list in zip(df_long.index, df_long["keywords_list"]):
    for kw in kw_list:
        col = f"kw_{kw}"
        if col in keyword_features.columns:
            keyword_features.loc[row_index, col] = 1

text_svd_features_raw = df_long[text_svd_cols].astype(float).fillna(0).copy()

print("Market features:", market_dummies.shape[1])
print("Origin features:", origin_dummies.shape[1])
print("Language features:", lang_dummies.shape[1])
print("Cluster features:", cluster_dummies.shape[1])
print("Genre features:", genre_dummies.shape[1])
print("Keyword features:", keyword_features.shape[1])
print("Budget bracket features:", bracket_dummies.shape[1])
print("Text SVD features:", text_svd_features_raw.shape[1])

Market features: 35
Origin features: 25
Language features: 24
Cluster features: 6
Genre features: 22
Keyword features: 470
Budget bracket features: 4
Text SVD features: 50


In [22]:
market_vals = market_dummies.values

cluster_domestic = pd.DataFrame(
    cluster_dummies.values * domestic_flag.values,
    columns=[f"{c}_x_domestic" for c in cluster_dummies.columns],
    index=df_long.index
)

lang_market = pd.DataFrame(
    (
        lang_dummies.values[:, :, None]
        * market_vals[:, None, :]
    ).reshape(len(df_long), -1),
    columns=[
        f"{language}_x_{market}"
        for language in lang_dummies.columns
        for market in market_dummies.columns
    ],
    index=df_long.index
)

lang_market = drop_sparse_cols(lang_market)

origin_market = pd.DataFrame(
    (
        origin_dummies.values[:, :, None]
        * market_vals[:, None, :]
    ).reshape(len(df_long), -1),
    columns=[
        f"{origin}_x_{market}"
        for origin in origin_dummies.columns
        for market in market_dummies.columns
    ],
    index=df_long.index
)

origin_market = drop_sparse_cols(origin_market)

cluster_market = pd.DataFrame(
    (
        cluster_dummies.values[:, :, None]
        * market_vals[:, None, :]
    ).reshape(len(df_long), -1),
    columns=[
        f"{cluster}_x_{market}"
        for cluster in cluster_dummies.columns
        for market in market_dummies.columns
    ],
    index=df_long.index
)

cluster_market = drop_sparse_cols(cluster_market)

kw_tier2_cols = [
    f"kw_{keyword}"
    for keyword in top_divisive_kw
    if f"kw_{keyword}" in keyword_features.columns
]

kw_tier2_matrix = keyword_features[kw_tier2_cols]

kw_market = pd.DataFrame(
    (
        kw_tier2_matrix.values[:, :, None]
        * market_vals[:, None, :]
    ).reshape(len(df_long), -1),
    columns=[
        f"{keyword}_x_{market}"
        for keyword in kw_tier2_cols
        for market in market_dummies.columns
    ],
    index=df_long.index
)

kw_market = drop_sparse_cols(kw_market)

print("Language × market:", lang_market.shape[1])
print("Origin × market:", origin_market.shape[1])
print("Cluster × market:", cluster_market.shape[1])
print("Cluster × domestic:", cluster_domestic.shape[1])
print("Keyword × market:", kw_market.shape[1])

Language × market: 68
Origin × market: 127
Cluster × market: 200
Cluster × domestic: 6
Keyword × market: 424


In [23]:
film_ids = df_long["titleId"].unique()

train_films, test_films = train_test_split(
    film_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_mask = df_long["titleId"].isin(train_films)
test_mask = df_long["titleId"].isin(test_films)

print("Train rows:", train_mask.sum())
print("Test rows:", test_mask.sum())
print("Train films:", len(train_films))
print("Test films:", len(test_films))

Train rows: 12642
Test rows: 3228
Train films: 851
Test films: 213


In [24]:
text_svd_features = text_svd_features_raw.copy()

text_train_mean = text_svd_features.loc[train_mask, text_svd_cols].mean()
text_train_std = text_svd_features.loc[train_mask, text_svd_cols].std().replace(0, 1)

text_svd_features[text_svd_cols] = (
    text_svd_features[text_svd_cols] - text_train_mean
) / text_train_std

print("Text SVD train means after scaling:")
print(text_svd_features.loc[train_mask, text_svd_cols].mean().round(4).head())

print("\nText SVD train stds after scaling:")
print(text_svd_features.loc[train_mask, text_svd_cols].std().round(4).head())

Text SVD train means after scaling:
text_svd_000   -0.0
text_svd_001    0.0
text_svd_002   -0.0
text_svd_003   -0.0
text_svd_004   -0.0
dtype: float64

Text SVD train stds after scaling:
text_svd_000    1.0
text_svd_001    1.0
text_svd_002    1.0
text_svd_003    1.0
text_svd_004    1.0
dtype: float64


In [25]:
base_with_brackets = pd.concat([
    market_dummies,
    origin_dummies,
    lang_dummies,
    cluster_dummies,
    domestic_flag,
    genre_dummies,
    keyword_features,
    bracket_dummies
], axis=1)

full_with_interactions = pd.concat([
    market_dummies,
    origin_dummies,
    lang_dummies,
    cluster_dummies,
    domestic_flag,
    genre_dummies,
    keyword_features,
    bracket_dummies,
    cluster_domestic,
    lang_market,
    origin_market,
    cluster_market,
    kw_market
], axis=1)

base_with_brackets_text = pd.concat([
    base_with_brackets,
    text_svd_features
], axis=1)

full_with_interactions_text = pd.concat([
    full_with_interactions,
    text_svd_features
], axis=1)

feature_versions = {
    "A_original_base_brackets": base_with_brackets,
    "B_original_full_interactions": full_with_interactions,
    "C_text_base_brackets": base_with_brackets_text,
    "D_text_full_interactions": full_with_interactions_text,
}

for name, features_df in feature_versions.items():
    print(f"{name}: {features_df.shape[1]} features")

A_original_base_brackets: 587 features
B_original_full_interactions: 1412 features
C_text_base_brackets: 637 features
D_text_full_interactions: 1462 features


In [26]:
results = []

for name, features_df in feature_versions.items():
    Xtr = csr_matrix(features_df.loc[train_mask].values.astype(np.float64))
    Xte = csr_matrix(features_df.loc[test_mask].values.astype(np.float64))

    print(f"\n{name}")

    for alpha in ALPHA_VALUES:
        model = Ridge(alpha=alpha, solver="sparse_cg")
        model.fit(Xtr, y[train_mask])

        train_r2 = model.score(Xtr, y[train_mask])
        test_r2 = model.score(Xte, y[test_mask])

        test_pred = model.predict(Xte)
        test_rmse = np.sqrt(mean_squared_error(y[test_mask], test_pred))

        results.append({
            "model_name": name,
            "alpha": alpha,
            "n_features": int(features_df.shape[1]),
            "train_r2": float(train_r2),
            "test_r2": float(test_r2),
            "test_rmse": float(test_rmse),
            "uses_text_svd": name.startswith("C_") or name.startswith("D_"),
            "uses_interactions": "full_interactions" in name,
        })

        print(
            f"alpha={alpha:<6} "
            f"Train R²={train_r2:.4f} "
            f"Test R²={test_r2:.4f} "
            f"Test RMSE={test_rmse:.4f}"
        )

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ["test_r2", "test_rmse"],
    ascending=[False, True]
).reset_index(drop=True)

results_df.to_csv(COMPARISON_OUTPUT_PATH, index=False)

print("\nBest results:")
print(results_df.head(10))

print("\nSaved comparison to:", COMPARISON_OUTPUT_PATH)


A_original_base_brackets
alpha=500    Train R²=0.4017 Test R²=0.1527 Test RMSE=0.9045
alpha=1000   Train R²=0.3537 Test R²=0.1762 Test RMSE=0.8919
alpha=2000   Train R²=0.3026 Test R²=0.1749 Test RMSE=0.8926
alpha=3000   Train R²=0.2721 Test R²=0.1668 Test RMSE=0.8969
alpha=5000   Train R²=0.2336 Test R²=0.1516 Test RMSE=0.9051
alpha=8000   Train R²=0.1984 Test R²=0.1342 Test RMSE=0.9143
alpha=10000  Train R²=0.1820 Test R²=0.1251 Test RMSE=0.9191

B_original_full_interactions
alpha=500    Train R²=0.4128 Test R²=0.1590 Test RMSE=0.9012
alpha=1000   Train R²=0.3608 Test R²=0.1798 Test RMSE=0.8900
alpha=2000   Train R²=0.3069 Test R²=0.1769 Test RMSE=0.8915
alpha=3000   Train R²=0.2753 Test R²=0.1682 Test RMSE=0.8962
alpha=5000   Train R²=0.2357 Test R²=0.1524 Test RMSE=0.9047
alpha=8000   Train R²=0.1999 Test R²=0.1347 Test RMSE=0.9141
alpha=10000  Train R²=0.1832 Test R²=0.1255 Test RMSE=0.9189

C_text_base_brackets
alpha=500    Train R²=0.4278 Test R²=0.1343 Test RMSE=0.9143
alpha=1

In [27]:
if AUTO_SELECT_BEST:
    best_row = results_df.iloc[0]
    BEST_MODEL_NAME = best_row["model_name"]
    BEST_ALPHA = float(best_row["alpha"])
else:
    BEST_MODEL_NAME = FORCED_MODEL_NAME
    BEST_ALPHA = FORCED_ALPHA

final_features = feature_versions[BEST_MODEL_NAME]

X_train_final = csr_matrix(
    final_features.loc[train_mask].values.astype(np.float64)
)

X_test_final = csr_matrix(
    final_features.loc[test_mask].values.astype(np.float64)
)

ridge_final = Ridge(alpha=BEST_ALPHA, solver="sparse_cg")
ridge_final.fit(X_train_final, y[train_mask])

train_r2 = ridge_final.score(X_train_final, y[train_mask])
test_r2 = ridge_final.score(X_test_final, y[test_mask])

test_predictions = ridge_final.predict(X_test_final)
test_rmse = np.sqrt(mean_squared_error(y[test_mask], test_predictions))

uses_text_svd_final = BEST_MODEL_NAME.startswith("C_") or BEST_MODEL_NAME.startswith("D_")
uses_interactions_final = "full_interactions" in BEST_MODEL_NAME

print("Final model name:", BEST_MODEL_NAME)
print("Uses text SVD:", uses_text_svd_final)
print("Uses interactions:", uses_interactions_final)
print("Final features:", final_features.shape[1])
print("Best alpha:", BEST_ALPHA)
print("Train R²:", round(train_r2, 4))
print("Test R²:", round(test_r2, 4))
print("Test RMSE:", round(test_rmse, 4))

Final model name: B_original_full_interactions
Uses text SVD: False
Uses interactions: True
Final features: 1412
Best alpha: 1000.0
Train R²: 0.3608
Test R²: 0.1798
Test RMSE: 0.89


In [28]:
predictions_df = df_long.loc[test_mask, [
    "titleId",
    "originalTitle",
    "country",
    "admissions",
    "log_admissions",
    "admissions_zscore",
    "mainCountry",
    "firstLanguage",
    "genres",
    "keywords",
    "cluster",
    "budget_eur",
    "budget_bracket",
    "is_domestic",
]].copy()

predictions_df["prediction"] = test_predictions
predictions_df["error"] = (
    predictions_df["prediction"]
    - predictions_df["admissions_zscore"]
)

predictions_df.to_csv(PREDICTIONS_OUTPUT_PATH, index=False)

print("Saved predictions to:", PREDICTIONS_OUTPUT_PATH)
print(predictions_df.head())

Saved predictions to: C:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\model_predictions_with_text_svd.csv
      titleId originalTitle country  admissions  log_admissions  \
36  tt1340107        Hævnen      AT        4400        8.389587   
37  tt1340107        Hævnen      BE       17294        9.758173   
38  tt1340107        Hævnen      CH       14254        9.564863   
39  tt1340107        Hævnen      CZ        4492        8.410276   
40  tt1340107        Hævnen      DE       91866       11.428097   

    admissions_zscore mainCountry firstLanguage          genres  \
36          -0.944047          DK            da  Drama, Romance   
37          -0.294725          DK            da  Drama, Romance   
38           0.048017          DK            da  Drama, Romance   
39          -0.933150          DK            da  Drama, Romance   
40          -0.228125          DK            da  Drama, Romance   

                                             keywords  cluster  

In [29]:
country_stats = (
    df_long.groupby("country")["log_admissions"]
    .agg(["mean", "std", "count"])
    .round(6)
)

country_stats["mean_raw"] = (
    df_long.groupby("country")["admissions"]
    .mean()
    .round(0)
)

country_stats["median_raw"] = (
    df_long.groupby("country")["admissions"]
    .median()
    .round(0)
)

print("Country stats for z-score to ticket conversion:")
print(country_stats)

if "DE" in country_stats.index:
    print("\nExample: z-score of +0.5 in DE")

    de_mean = country_stats.loc["DE", "mean"]
    de_std = country_stats.loc["DE", "std"]

    log_est = 0.5 * de_std + de_mean
    ticket_est = np.expm1(log_est)

    print(f"log_admissions = 0.5 * {de_std:.3f} + {de_mean:.3f} = {log_est:.3f}")
    print(f"estimated tickets = {ticket_est:,.0f}")

Country stats for z-score to ticket conversion:
              mean       std  count  mean_raw  median_raw
country                                                  
AT       10.062208  1.771756    527   79803.0     28078.0
BA        7.602298  1.686576    170    5371.0      2174.0
BE       10.304936  1.855162    600  104516.0     35692.0
BG        9.405138  1.605805    359   29425.0     15568.0
CH        9.441277  2.573793    673   74743.0     24546.0
CY        7.719719  1.525526    202    6698.0      2242.0
CZ        9.976504  1.678431    510   62578.0     24204.0
DE       11.896114  2.051584    631  658232.0    178493.0
DK        9.841874  2.023624    536   80450.0     24634.0
EE        8.628550  1.298936    440   11529.0      5818.0
ES       11.009153  2.987352    802  440119.0    136394.0
FI        9.599917  1.817953    478   49151.0     17910.0
FR       11.800820  2.365912    791  748163.0    204439.0
GB       10.820525  3.240899    244  747717.0     99636.0
GB_IE    11.605134  2.78

In [30]:
def categorize_feature(name):
    if name.startswith("text_svd_"):
        return "text svd"

    if "_x_market_" in name:
        if name.startswith("lang_"):
            return "language x market"
        if name.startswith("origin_"):
            return "origin x market"
        if name.startswith("kw_"):
            return "keyword x market"
        if name.startswith("cluster_"):
            return "cluster x market"

    if "_x_domestic" in name:
        return "cluster x domestic"

    if name.startswith("market_"):
        return "market baseline"

    if name.startswith("origin_"):
        return "origin"

    if name.startswith("lang_"):
        return "language"

    if name.startswith("cluster_"):
        return "cluster"

    if name.startswith("genre_"):
        return "genre"

    if name.startswith("kw_"):
        return "keyword"

    if name.startswith("budget_"):
        return "budget bracket"

    if name == "is_domestic":
        return "domestic flag"

    return "other"


feature_category_counts = Counter(
    categorize_feature(feature)
    for feature in final_features.columns
)

print(feature_category_counts)

Counter({'keyword': 470, 'keyword x market': 424, 'cluster x market': 200, 'origin x market': 127, 'language x market': 68, 'market baseline': 35, 'origin': 25, 'language': 24, 'genre': 22, 'cluster': 6, 'cluster x domestic': 6, 'budget bracket': 4, 'domestic flag': 1})


In [32]:
coef_dict = {}

for feature, coef in zip(final_features.columns, ridge_final.coef_):
    if abs(coef) > 0.0001:
        coef_dict[feature] = round(float(coef), 6)

available_clusters = []

for col in final_features.columns:
    if col.startswith("cluster_") and "_x_" not in col:
        cluster_value = str(col).replace("cluster_", "")

        try:
            available_clusters.append(int(float(cluster_value)))
        except ValueError:
            available_clusters.append(cluster_value)

cluster_descriptions = {
    "0": "Horror / Thriller / Supernatural",
    "1": "Serious Drama / War / Historical",
    "2": "Buddy Comedy / Coming-of-age",
    "3": "Crime Thriller / Detective",
    "4": "Romance / Relationship Drama",
    "5": "Adventure / Fantasy / Family",
}

export_data = {
    "model_info": {
        "model_name": BEST_MODEL_NAME,
        "alpha": BEST_ALPHA,
        "n_features": int(final_features.shape[1]),
        "n_nonzero_coefs": len(coef_dict),
        "intercept": round(float(ridge_final.intercept_), 6),
        "train_r2": round(float(train_r2), 4),
        "test_r2": round(float(test_r2), 4),
        "test_rmse": round(float(test_rmse), 4),
        "n_train_films": int(len(train_films)),
        "n_test_films": int(len(test_films)),
        "uses_text_svd": bool(uses_text_svd_final),
        "uses_interactions": bool(uses_interactions_final),
        "text_svd_features": text_svd_cols if uses_text_svd_final else [],
    },
    "feature_category_counts": {
        key: int(value)
        for key, value in feature_category_counts.items()
    },
    "coef_dict": coef_dict,
    "countries": sorted(df_long["country"].unique().tolist()),
    "country_stats": {
        country: {
            "mean_log": round(float(row["mean"]), 6),
            "std_log": round(float(row["std"]), 6),
            "mean_raw": float(row["mean_raw"]),
            "median_raw": float(row["median_raw"]),
            "n_films": int(row["count"]),
        }
        for country, row in country_stats.iterrows()
    },
    "available_keywords": sorted([
        col.replace("kw_", "")
        for col in final_features.columns
        if col.startswith("kw_") and "_x_" not in col
    ]),
    "available_genres": sorted([
        col.replace("genre_", "")
        for col in final_features.columns
        if col.startswith("genre_")
    ]),
    "available_languages": sorted([
        col.replace("lang_", "")
        for col in final_features.columns
        if col.startswith("lang_")
    ]),
    "available_origins": sorted([
        col.replace("origin_", "")
        for col in final_features.columns
        if col.startswith("origin_")
    ]),
    "available_clusters": sorted(available_clusters),
    "budget_brackets": sorted([
        col.replace("budget_", "")
        for col in final_features.columns
        if col.startswith("budget_")
    ]),
    "cluster_descriptions": cluster_descriptions,
    "interaction_keywords": top_divisive_kw,
    "text_svd_note": (
        "This export includes TF-IDF/SVD text features if uses_text_svd is true. "
        "For prediction on new unseen films, the same TF-IDF/SVD transformation must be available."
    ),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(EXPORT_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2)

print("Exported to:", EXPORT_OUTPUT_PATH)
print("File size:", round(EXPORT_OUTPUT_PATH.stat().st_size / 1024, 1), "KB")

print("\nContents:")
print("Final model:", BEST_MODEL_NAME)
print("Test R²:", round(test_r2, 4))
print("Test RMSE:", round(test_rmse, 4))
print("Non-zero coefficients:", len(coef_dict))
print("Countries:", len(export_data["countries"]))
print("Keywords:", len(export_data["available_keywords"]))
print("Genres:", len(export_data["available_genres"]))
print("Languages:", len(export_data["available_languages"]))
print("Origins:", len(export_data["available_origins"]))
print("Text SVD features used:", uses_text_svd_final)

Exported to: C:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\model_export_with_text_svd.json
File size: 73.5 KB

Contents:
Final model: B_original_full_interactions
Test R²: 0.1798
Test RMSE: 0.89
Non-zero coefficients: 1399
Countries: 35
Keywords: 470
Genres: 22
Languages: 92
Origins: 152
Text SVD features used: False


In [33]:
summary_cols = [
    "model_name",
    "alpha",
    "n_features",
    "train_r2",
    "test_r2",
    "test_rmse",
    "uses_text_svd",
    "uses_interactions",
]

print("Top 10 model configurations:")
display(results_df[summary_cols].head(10))

original_best = (
    results_df[results_df["uses_text_svd"] == False]
    .sort_values(["test_r2", "test_rmse"], ascending=[False, True])
    .iloc[0]
)

text_best = (
    results_df[results_df["uses_text_svd"] == True]
    .sort_values(["test_r2", "test_rmse"], ascending=[False, True])
    .iloc[0]
)

r2_change = text_best["test_r2"] - original_best["test_r2"]
rmse_change = text_best["test_rmse"] - original_best["test_rmse"]

print("\nBest original model:")
print(original_best[summary_cols])

print("\nBest text-extended model:")
print(text_best[summary_cols])

print("\nText SVD effect:")
print(f"R² change: {r2_change:.4f}")
print(f"RMSE change: {rmse_change:.4f}")

if r2_change > 0 and rmse_change < 0:
    print("\nAdding TF-IDF/SVD text features improved the Julius model.")
elif r2_change > 0:
    print("\nAdding TF-IDF/SVD text features improved R², but RMSE should also be checked.")
else:
    print("\nAdding TF-IDF/SVD text features did not improve the model in this run.")

Top 10 model configurations:


,model_name,alpha,n_features,train_r2,test_r2,test_rmse,uses_text_svd,uses_interactions
0,B_original_full_interactions,1000,1412,0.360765,0.179764,0.889958,False,True
1,B_original_full_interactions,2000,1412,0.306925,0.176895,0.891513,False,True
2,A_original_base_brackets,1000,587,0.353684,0.176167,0.891907,False,False
3,A_original_base_brackets,2000,587,0.302609,0.174893,0.892596,False,False
4,B_original_full_interactions,3000,1412,0.275288,0.168208,0.896205,False,True
5,A_original_base_brackets,3000,587,0.272126,0.166835,0.896944,False,False
6,D_text_full_interactions,2000,1462,0.337242,0.165117,0.897869,True,True
7,C_text_base_brackets,2000,637,0.332919,0.163215,0.898891,True,False
8,D_text_full_interactions,1000,1462,0.388808,0.162708,0.899163,True,True
9,D_text_full_interactions,3000,1462,0.306777,0.160067,0.900580,True,True



Best original model:
model_name           B_original_full_interactions
alpha                                        1000
n_features                                   1412
train_r2                                 0.360765
test_r2                                  0.179764
test_rmse                                0.889958
uses_text_svd                               False
uses_interactions                            True
Name: 0, dtype: object

Best text-extended model:
model_name           D_text_full_interactions
alpha                                    2000
n_features                               1462
train_r2                             0.337242
test_r2                              0.165117
test_rmse                            0.897869
uses_text_svd                            True
uses_interactions                        True
Name: 6, dtype: object

Text SVD effect:
R² change: -0.0146
RMSE change: 0.0079

Adding TF-IDF/SVD text features did not improve the model in this run.
